Данный ноутбук представляет собой выполненную работу в рамках летней практики СПБГУ от VK.
    
**Тема**:   Обучение визуально-языковых моделей

**Описание**:   Исследование методов борьбы с феноменом *critical forgetting* на примере дообученной на датасете
ruCommonVQA модели от vk *llava-gemma-2b-lora*. Изначально модель дообучается с помощью механизма *QLora*(параметры см. Qlora_config), проводится сравнение метрики *rouge* на *GQA-ru* до и после дообучения. Далее исследуется эффективность механизмов дообучения VLM, разработанных для борьбы с critical forgetting: . Оригинальные статьи указаны в цитатах.

**Выполнил:** Кайманов К. студент 2 курса ПМ-ПУ СПБГУ

In [2]:

import numpy as np
import pandas as pd
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from datasets import load_dataset
import kagglehub
import logging
import torch
import PIL
from PIL import Image
from datasets import load_dataset, Dataset
from io import BytesIO

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
#задание логгера, чтобы видеть скорость выполнения скрипта. 
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Препроцессинг. Подготовка датасета.

In [3]:

ru_common = "ru_common" 
gqa = "GQA"

''' Переменные для обозначения датасетов, чтобы не писать под каждый отдельную load_ds. Общий механизм подгрузки 
у них схожий. '''

' Переменные для обозначения датасетов, чтобы не писать под каждый отдельную load_ds. Общий механизм подгрузки \nу них схожий. '

In [4]:
def format_to_gemma_style(conversation: list) -> dict: #lllava chchchicken
    human = conversation[0]
    gpt = conversation[1]
    return {"conversation": {"role": "user", "content": f"<image>\n{human}"}, "answer" : gpt}

def load_image_from_url(sample):
    img = Image.open(BytesIO(sample["bytes"]))
    buffer = BytesIO()
    img.save(buffer, format='JPEG', quality=60, optimize=True) #слишком долго собирался датасет без оптимизации
    
    return Image.open(buffer)

def load_ds(dset = ru_common) -> tuple:
    if dset == ru_common:
        logger.info("загрузка данных...")
        dataset = load_dataset("MERA-evaluation/ruCommonVQA", split="test")
        logger.info("создание индекса изображений...")
        dataset_length = len(dataset["meta"])
        log_step = dataset_length // 20
        index_image = dict() #тот самый индекс
        for i in range(dataset_length):
            index_image[dataset["meta"][i]["id"]] = dataset["inputs"][i]["image"]

        logger.info("объединение, форматирование датасета под llava...")
        train_data = []
    
        for j in range(dataset_length):
            image_id = dataset["meta"][j]["id"]
            if (j + 1) % log_step == 0 or (j + 1) == dataset_length:
                percent = (j + 1) / dataset_length * 100
                logger.info(f"Прогресс: {j + 1}/{dataset_length} ({percent:.1f}%)")
            #для упрощения взят только один вариант ответа - первый.
            llava_style_instructions = format_to_gemma_style([dataset["inputs"][j]["question"], dataset["outputs"][j][0]])
            train_data.append({"image":index_image[image_id], "instruction": llava_style_instructions["conversation"], "answer": llava_style_instructions["answer"]})
            
    elif dset == gqa:
        
        logger.info("загрузка данных...")
    
        image_dataset = load_dataset("deepvk/GQA-ru", "testdev_balanced_images", streaming=True)
        text_dataset = load_dataset("deepvk/GQA-ru", "testdev_balanced_instructions", streaming=True)

        image_dataset = image_dataset["testdev"]
        text_dataset = text_dataset["testdev"]
    

        logger.info("объединение, форматирование датасета под llava...")
    
        train_data = []
        for text in text_dataset:
            image_id = text["imageId"]
            train_data.append({"image":image_id, "instruction": format_to_gemma_style([text["question"], text["answer"]])})


    dataset = Dataset.from_list(train_data)
    alpha = 0.9
    split_idx = int(len(train_data) * alpha)
    train_ds = dataset.select(range(split_idx))
    eval_ds = dataset.select(range(split_idx, len(dataset)))
    logger.info("Готово!")
    if dset == ru_common:
        return train_ds, eval_ds
    elif dset == gqa:
        return train_ds, eval_ds, image_dataset
    
#load_ds()

Создание data collator. Загрузка модели.

In [5]:
class DataCollator:
    def __init__(self, processor):
        self.processor = processor
        self.max_seq_length = 1024 #простите, лень замерять точно

    
    def __call__(self, batch):
        
        prompts = []
        answers = []
        images = []
        
        for element in batch:
            element["image"] = PIL.Image.open(BytesIO(element["image"]["bytes"])).convert("RGB")
            prompts.append(element["instruction"]["content"])
            answers.append(element["answer"])
            images.append(element["image"])

        prompt_tokens = self.processor.tokenizer(prompts, padding=False, add_special_tokens=False, return_tensors=None)
        prompt_lengths = [len(tokens) for tokens in prompt_tokens['input_ids']]

        full_text = [p + a for p, a in zip(prompts, answers)]

        inputs = self.processor(text = full_text, images = images, padding = True, truncation=True, max_length = self.max_seq_length, return_tensors="pt")

        labels = inputs['input_ids'].clone()
        for i, prompt_len in enumerate(prompt_lengths):
            labels[i, :prompt_len] = -100 #скрытие промпта, чтобы кросс-энтропия его не учитывала
        
        '''device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        for k, v in inputs.items():
            if isinstance(v, torch.Tensor):
                inputs[k] = v.to(device)        
        inputs['labels'] = labels.to(device)'''
        inputs['labels'] = labels

        return inputs

  Теперь, когда есть готовый механизм для подгрузки датасета и коллатор данных, можем дообучить модель на ruCommonVQA. Однако, для начала нужно проверить rouge as-is модели на GQA-ru test. Если метрика после tuning-а просядет, это покажет наличие феномена critical forgetting.

In [6]:
!rm -rf /opt/conda/lib/python3.10/site-packages/aiohttp* && pip install aiohttp trl 
!pip install evaluate
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 15.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00:00:0100:01


In [7]:
import evaluate
import torch
from transformers import AutoModel
from transformers import AutoProcessor, AutoTokenizer
from transformers import enable_full_determinism
from transformers import BitsAndBytesConfig
from transformers import AutoModelForMultimodalLM
rouge = evaluate.load("rouge") #фактически метрика анализирует вхождение слов в ответ или полное совпадение.
model_name = "deepvk/llava-gemma-2b-lora" #использованы материалы вк
enable_full_determinism(42)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForMultimodalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="balanced",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
processor = AutoProcessor.from_pretrained(
    model_name,
    patch_size=14,
    trust_remote_code=True,
)
processor.image_processor.size = {"shortest_edge": 224, "longest_edge": 224}
print(torch.cuda.memory_summary())

2026-07-14 23:55:34,978 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:55:34,980 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-07-14 23:55:34,992 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/config.json "HTTP/1.1 200 OK"
2026-07-14 23:55:35,006 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/config.json "HTTP/1.1 200 OK"


config.json: 0.00B [00:00, ?B/s]

2026-07-14 23:55:35,104 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-07-14 23:55:35,219 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:55:35,231 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/config.json "HTTP/1.1 200 OK"
2026-07-14 23:55:40,159 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-14 23:55:40,244 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/model.safetensors.index.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:55:40,256 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbf

model.safetensors.index.json: 0.00B [00:00, ?B/s]

2026-07-14 23:55:40,365 - INFO - HTTP Request: GET https://huggingface.co/api/models/deepvk/llava-gemma-2b-lora/revision/main "HTTP/1.1 200 OK"


...

processor_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

2026-07-14 23:56:38,252 - INFO - HTTP Request: GET https://huggingface.co/api/models/deepvk/llava-gemma-2b-lora/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-14 23:56:38,338 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:56:38,350 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/processor_config.json "HTTP/1.1 200 OK"
2026-07-14 23:56:38,439 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-07-14 23:56:38,527 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-07-14 23:56:38,615 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lor

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
2026-07-14 23:56:39,386 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:56:39,399 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d674b580fcbfcdc6b1479a1f95c429b24a/processor_config.json "HTTP/1.1 200 OK"
2026-07-14 23:56:39,489 - INFO - HTTP Request: HEAD https://huggingface.co/deepvk/llava-gemma-2b-lora/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:56:39,503 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepvk/llava-gemma-2b-lora/30f1c8d6

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3133 MiB |   9935 MiB |  15239 MiB |  12106 MiB |
|       from large pool |   3039 MiB |   9929 MiB |  15087 MiB |  12048 MiB |
|       from small pool |     94 MiB |     94 MiB |    152 MiB |     57 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   3133 MiB |   9935 MiB |  15239 MiB |  12106 MiB |
|       from large pool |   3039 MiB |   9929 MiB |  15087 MiB |

In [ ]:
gqa_dataset = load_ds(gqa)

In [ ]:
import time
from IPython.display import Image, display
import io
import re
image_dataset = gqa_dataset[2]
    
model.eval()


def normalize(text):
    text = re.sub(r'[^\w\s]', '', text)
    return text.lower().strip()


def get_image(image_id):
    for image in image_dataset:
            if image_id == image["id"]:
                return image["image"]
                

def predict(model, processor, image, prompt, tokenizer):
    #Генерация ответа модели
    
    
    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        print(f"Имя GPU: {torch.cuda.get_device_name(0)}")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=7,          
            do_sample=False,           
        )
    
    generated_text = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:], 
        skip_special_tokens=True
    )
    del inputs, outputs
    torch.cuda.empty_cache()
    
    return generated_text


def predict_batch(model, processor, images, prompts, max_new_tokens=10):
    texts = []
    for i in range(len(prompts)):
         text = tokenizer.apply_chat_template([prompts[i]], tokenize=False, add_generation_prompt=True)
         texts.append(text)

    
    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id
        )
    
    responses = []
    input_length = inputs['input_ids'].shape[1]
    for i in range(len(images)):
        response = processor.decode(
            outputs[i][input_length:],
            skip_special_tokens=True
        )
        responses.append(response)
    del inputs, outputs
    torch.cuda.empty_cache()
    
    return responses


predicted_answers = []
true_answers = []
short_phrase = " Ответь кратко."
gqa_train = gqa_dataset[1]
print(gqa_train)
ln = len(gqa_train)
'''for element in gqa_train:
    i += 1
    image = get_image(element["image"])
    conversation = str(element["instruction"])
    answer_index = conversation.find("ASSISTANT: ") + len("ASSISTANT: ")
    prompt = conversation[:answer_index]
    answer = conversation[answer_index:]
    print(image)
    print(prompt)
    start = time.time()
    predicted_answer = predict(model, processor, image, prompt)
    end = time.time()
    print("ВРЕМЯ:", end - start)
    predicted_answers.append(predicted_answer)
    true_answers.append(answer)'''

def get_rouge(batch_size, ln, model):
    for i in range(0, ln, batch_size):
        batch_prompts = []
        batch_images = []
        batch_true = []
        for j in range(i, min(i + batch_size, ln)):
            image = get_image(gqa_train["image"][j])
            
            please_shorter = gqa_train["instruction"][j]["conversation"]
            please_shorter["content"] += short_phrase
            prompt =  please_shorter
            
            true_answer = gqa_train["instruction"][j]["answer"]
            #print(prompt, true_answer)
            #buffered = io.BytesIO()
            #image.save(buffered, format="PNG")
            #display(Image(data=buffered.getvalue()))
            batch_prompts.append(prompt)
            batch_images.append(image)
            batch_true.append(true_answer)
        
        print(str(i / ln) + "%", "обработано")
        batch_predictions = predict_batch(model, processor, batch_images, batch_prompts)
        #print(batch_predictions)
        predicted_answers.extend(batch_predictions)
        for p in range(min(batch_size, ln - p)):
            if batch_predictions[p][:2] == "\n":
                batch_predictions[p] = batch_predictions[2:]
        true_answers.extend(batch_true)

    
    true_norm = [normalize(t) for t in true_answers]
    pred_norm = [normalize(p) for p in predicted_answers]
    for i, (p, r) in enumerate(zip(pred_norm, true_norm)):
        print(f"Pair {i}: pred='{p}', ref='{r}'")
        
    rouge_asis = rouge.compute(
        predictions=pred_norm,
        references=true_norm,
        tokenizer = lambda x: x.split() #rouge, как оказывается, из коробки слабо поддерживает русский
        )
    return rouge_asis

get_rouge(3, len(gqa_train), model)   

rouge1, rougeLsum: **0.44**. Скорее всего, падение метрики относительно представленной в материалах вк связано с использованием четырехбитной квантизации и недостаточно отлаженного пайплайна обработки промптов, однако в пределах ожидаемого. Можно приступать к дообучению модели.

In [8]:

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

train_ds, eval_ds = load_ds()


2026-07-14 23:56:57,416 - INFO - загрузка данных...
2026-07-14 23:56:57,549 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/MERA-evaluation/ruCommonVQA/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-07-14 23:56:57,562 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/MERA-evaluation/ruCommonVQA/8b6f292be8c2775e377f788115d7828c1a5f7d1e/README.md "HTTP/1.1 200 OK"
2026-07-14 23:56:57,575 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/MERA-evaluation/ruCommonVQA/8b6f292be8c2775e377f788115d7828c1a5f7d1e/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

2026-07-14 23:56:57,670 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/MERA-evaluation/ruCommonVQA/resolve/8b6f292be8c2775e377f788115d7828c1a5f7d1e/ruCommonVQA.py "HTTP/1.1 404 Not Found"
2026-07-14 23:56:57,923 - INFO - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/MERA-evaluation/ruCommonVQA/MERA-evaluation/ruCommonVQA.py "HTTP/1.1 404 Not Found"
2026-07-14 23:56:58,117 - INFO - HTTP Request: GET https://huggingface.co/api/datasets/MERA-evaluation/ruCommonVQA/revision/8b6f292be8c2775e377f788115d7828c1a5f7d1e "HTTP/1.1 200 OK"
2026-07-14 23:56:58,197 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/MERA-evaluation/ruCommonVQA/resolve/8b6f292be8c2775e377f788115d7828c1a5f7d1e/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-07-14 23:56:58,323 - INFO - HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=MERA-evaluation/ruCommonVQA "HTTP/1.1 503 Service Temporarily Unavailable"
2026-07-14 23:56:58,440 - INFO 

data/shots-00000-of-00001.parquet:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

2026-07-14 23:56:59,306 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/MERA-evaluation/ruCommonVQA/resolve/8b6f292be8c2775e377f788115d7828c1a5f7d1e/data/test-00000-of-00003.parquet "HTTP/1.1 302 Found"


...

2026-07-14 23:57:03,734 - INFO - HTTP Request: HEAD https://huggingface.co/datasets/MERA-evaluation/ruCommonVQA/resolve/8b6f292be8c2775e377f788115d7828c1a5f7d1e/data/test-00002-of-00003.parquet "HTTP/1.1 302 Found"


data/test-00002-of-00003.parquet:   0%|          | 0.00/405M [00:00<?, ?B/s]

Generating shots split:   0%|          | 0/10 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3015 [00:00<?, ? examples/s]

2026-07-14 23:57:08,686 - INFO - создание индекса изображений...
2026-07-14 23:57:14,776 - INFO - объединение, форматирование датасета под llava...
2026-07-14 23:57:15,160 - INFO - Прогресс: 150/3015 (5.0%)
2026-07-14 23:57:15,545 - INFO - Прогресс: 300/3015 (10.0%)
2026-07-14 23:57:15,943 - INFO - Прогресс: 450/3015 (14.9%)
2026-07-14 23:57:16,334 - INFO - Прогресс: 600/3015 (19.9%)
2026-07-14 23:57:16,717 - INFO - Прогресс: 750/3015 (24.9%)
2026-07-14 23:57:17,118 - INFO - Прогресс: 900/3015 (29.9%)
2026-07-14 23:57:17,540 - INFO - Прогресс: 1050/3015 (34.8%)
2026-07-14 23:57:17,928 - INFO - Прогресс: 1200/3015 (39.8%)
2026-07-14 23:57:18,310 - INFO - Прогресс: 1350/3015 (44.8%)
2026-07-14 23:57:18,670 - INFO - Прогресс: 1500/3015 (49.8%)
2026-07-14 23:57:19,013 - INFO - Прогресс: 1650/3015 (54.7%)
2026-07-14 23:57:19,365 - INFO - Прогресс: 1800/3015 (59.7%)
2026-07-14 23:57:19,710 - INFO - Прогресс: 1950/3015 (64.7%)
2026-07-14 23:57:20,052 - INFO - Прогресс: 2100/3015 (69.7%)
2026-

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./llava-gemma-lora-sft",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    dataloader_num_workers = 2,
    bf16 = False,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    report_to=[],
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollator(processor),
    processing_class=processor.tokenizer,
)

trainer.train()

trainer.save_model("./llava-gemma-simple-lora")
processor.save_pretrained("./llava-gemma-simple")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


trainable params: 21,970,944 || all params: 2,838,077,440 || trainable%: 0.7741


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,13.029514,12.779972


In [10]:
import zipfile
import os

def zip_folder(folder_path, zip_name):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                zipf.write(os.path.join(root, file),
                           os.path.relpath(os.path.join(root, file),
                                           os.path.join(folder_path, '..')))

zip_folder('./llava-gemma-simple-lora', 'lora-adapter.zip')
zip_folder('./llava-gemma-simple', 'processor.zip')

In [ ]:
'''import zipfile
import os

zip_path = "/kaggle/input/название-вашего-датасета/lora-adapter.zip"

extract_path = "/kaggle/working/llava-gemma-simple-lora"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)'''

#распаковать зип-архив с Lora

In [ ]:
'''base_model = LlavaForConditionalGeneration.from_pretrained(
    "deepvk/llava-gemma-2b-lora",
    device_map="auto",
    torch_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained("deepvk/llava-gemma-2b-lora")

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/llava-gemma-simple-lora" 
)

model.eval()'''

#использовать для инференса

Теперь, когда модель успешно дообучена, посчитаем rouge еще раз

In [ ]:
after_learning_rouge = get_rouge(4, 10, model) #вместо 10 подсавить длину датасета для оценки(1223)
after_learning_rouge

Имеет ли место critical forgetting?

*Оптимизация QLora*

Источник: https://huggingface.co/datasets/John6666/forum1/blob/903d17a29f34d7035a22abcde6936569adc2fbf1/lora_ft_params.md?code=true

*Experience Replay*

Источник: Rebuffi et al., 2017. "iCaRL: Incremental Classifier and Representation Learning"

*L2-LoRA*

Источник: Zhang et al., 2026. "L2-LoRA: Improving Low-Rank Adaptation with Layer-Specific Regularization" (AAAI 2026).

